# Tugas 2 - Perbandingan CountVectorizer dan TF-IDF pada Naive Bayes

Pada tugas ini digunakan dataset `spam.csv` yang berisi pesan SMS dengan label **spam** dan **ham**. Model yang digunakan adalah **Multinomial Naive Bayes** dengan dua teknik ekstraksi fitur, yaitu **CountVectorizer** dan **TF-IDF**, yang keduanya mengaktifkan **stop_words** bahasa Inggris.

## 1. Load Data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cari dataset: folder lokal -> /content (Colab) -> Google Drive -> GitHub
GITHUB_URL = 'https://raw.githubusercontent.com/oninonins/Machine-learning-/main/JS09/spam.csv'
DATA_CANDIDATES = [
    'spam.csv',
    '/content/spam.csv',
    '/content/drive/MyDrive/spam.csv',
    GITHUB_URL,
]

DATA_PATH = next(
    (p for p in DATA_CANDIDATES
     if not p.startswith('http') and os.path.exists(p)),
    None,
)

if DATA_PATH is None:
    DATA_PATH = GITHUB_URL

try:
    df = pd.read_csv(DATA_PATH, encoding='latin-1')
except Exception as exc:
    raise FileNotFoundError(
        'spam.csv tidak ditemukan. Letakkan file di folder notebook, '
        'upload ke /content, atau sesuaikan DATA_CANDIDATES.'
    ) from exc

print('Dataset:', DATA_PATH)
df.head()

## 2. Preprocessing

Tiga kolom terakhir dibuang karena tidak digunakan, lalu kolom `v1` (label) dan `v2` (teks SMS) diubah namanya agar lebih mudah dibaca.

In [ ]:
# drop 3 kolom terakhir
df = df.drop(df.iloc[:, 2:], axis=1)

# rename kolom
df = df.rename(columns={'v1': 'Labels', 'v2': 'SMS'})
df.head()

In [ ]:
# inspeksi data
print(df['Labels'].value_counts())
print('\n')
print(df.info())

## 3. Encoding Label dan Pemisahan Data

Label diubah menjadi angka (`spam = 1`, `ham = 0`), kemudian data dibagi 80% latih dan 20% uji dengan `random_state=50` agar konsisten dengan Praktikum 3.

In [ ]:
# encode label
df['Labels'] = df['Labels'].map({'spam': 1, 'ham': 0})

X = df['SMS'].values
y = df['Labels'].values

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=50
)

print('Jumlah data latih:', len(X_train))
print('Jumlah data uji  :', len(X_test))

## 4. Model Multinomial Naive Bayes dengan CountVectorizer

`CountVectorizer(stop_words='english')` menghitung frekuensi kemunculan kata setelah kata umum bahasa Inggris dibuang. Vectorizer hanya di-*fit* pada data latih untuk mencegah information leakage.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# ekstraksi fitur CountVectorizer
bow = CountVectorizer(stop_words='english')
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

# training model
mnb_bow = MultinomialNB()
mnb_bow.fit(X_train_bow, y_train)

# prediksi
y_train_pred_bow = mnb_bow.predict(X_train_bow)
y_test_pred_bow = mnb_bow.predict(X_test_bow)

print('Jumlah kosakata:', len(bow.get_feature_names_out()))
print('Akurasi train :', accuracy_score(y_train, y_train_pred_bow))
print('Akurasi test  :', accuracy_score(y_test, y_test_pred_bow))
print('Precision spam:', precision_score(y_test, y_test_pred_bow))
print('Recall spam   :', recall_score(y_test, y_test_pred_bow))
print('F1-score spam :', f1_score(y_test, y_test_pred_bow))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_test_pred_bow))
print('\nClassification Report:\n', classification_report(
    y_test, y_test_pred_bow, target_names=['ham', 'spam']
))

## 5. Model Multinomial Naive Bayes dengan TF-IDF

`TfidfVectorizer(stop_words='english')` memberi bobot pada setiap kata berdasarkan frekuensi kemunculan dan seberapa jarang kata tersebut muncul di seluruh dokumen.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# ekstraksi fitur TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# training model
mnb_tfidf = MultinomialNB()
mnb_tfidf.fit(X_train_tfidf, y_train)

# prediksi
y_train_pred_tfidf = mnb_tfidf.predict(X_train_tfidf)
y_test_pred_tfidf = mnb_tfidf.predict(X_test_tfidf)

print('Jumlah kosakata:', len(tfidf.get_feature_names_out()))
print('Akurasi train :', accuracy_score(y_train, y_train_pred_tfidf))
print('Akurasi test  :', accuracy_score(y_test, y_test_pred_tfidf))
print('Precision spam:', precision_score(y_test, y_test_pred_tfidf))
print('Recall spam   :', recall_score(y_test, y_test_pred_tfidf))
print('F1-score spam :', f1_score(y_test, y_test_pred_tfidf))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_test_pred_tfidf))
print('\nClassification Report:\n', classification_report(
    y_test, y_test_pred_tfidf, target_names=['ham', 'spam']
))

## 6. Perbandingan Hasil

In [ ]:
hasil = pd.DataFrame({
    'Fitur': ['CountVectorizer', 'TF-IDF'],
    'Akurasi Test': [
        accuracy_score(y_test, y_test_pred_bow),
        accuracy_score(y_test, y_test_pred_tfidf)
    ],
    'Precision Spam': [
        precision_score(y_test, y_test_pred_bow),
        precision_score(y_test, y_test_pred_tfidf)
    ],
    'Recall Spam': [
        recall_score(y_test, y_test_pred_bow),
        recall_score(y_test, y_test_pred_tfidf)
    ],
    'F1-score Spam': [
        f1_score(y_test, y_test_pred_bow),
        f1_score(y_test, y_test_pred_tfidf)
    ]
})

hasil

In [ ]:
# grafik perbandingan metrik
hasil.set_index('Fitur').plot(kind='bar', figsize=(9, 5))
plt.title('Perbandingan CountVectorizer dan TF-IDF')
plt.ylabel('Nilai')
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()

In [ ]:
# confusion matrix kedua model
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.heatmap(confusion_matrix(y_test, y_test_pred_bow), annot=True, fmt='d',
            cmap='Blues', ax=axes[0], xticklabels=['ham', 'spam'],
            yticklabels=['ham', 'spam'])
axes[0].set_title('CountVectorizer')
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Aktual')

sns.heatmap(confusion_matrix(y_test, y_test_pred_tfidf), annot=True, fmt='d',
            cmap='Blues', ax=axes[1], xticklabels=['ham', 'spam'],
            yticklabels=['ham', 'spam'])
axes[1].set_title('TF-IDF')
axes[1].set_xlabel('Prediksi')
axes[1].set_ylabel('Aktual')

plt.tight_layout()
plt.show()

## 7. Analisis Hasil dan Kesimpulan

### Perbandingan metrik

| Metrik | CountVectorizer | TF-IDF |
|---|---|---|
| Akurasi train | 0.9946 | 0.9843 |
| Akurasi test | **0.9830** | 0.9605 |
| Precision spam | 0.9797 | **1.0000** |
| Recall spam | **0.9006** | 0.7267 |
| F1-score spam | **0.9385** | 0.8417 |

### Pembahasan

1. **CountVectorizer** menghasilkan akurasi data uji sebesar **0.9830**, lebih tinggi dibandingkan TF-IDF yang memperoleh **0.9605**. CountVectorizer juga lebih baik dalam mendeteksi kelas spam, terlihat dari recall spam **0.9006** dan F1-score **0.9385**.
2. **TF-IDF** memang menghasilkan precision spam sempurna (**1.0000**), artinya semua SMS yang diprediksi spam benar-benar spam. Namun recall-nya hanya **0.7267**, sehingga sekitar 27% SMS spam gagal terdeteksi. Model TF-IDF terlalu konservatif dalam memberi label spam.
3. Confusion matrix menunjukkan CountVectorizer hanya melewatkan **16** SMS spam, sedangkan TF-IDF melewatkan **44** SMS spam. Dari sisi keamanan penyaringan spam, melewatkan spam (false negative) umumnya lebih merugikan daripada salah menandai SMS normal sebagai spam.
4. Pada Multinomial Naive Bayes, fitur berupa **frekuensi kata mentah** lebih sesuai dengan asumsi model multinomial. TF-IDF mengubah nilai fitur menjadi bobot kontinu dan menurunkan pengaruh kata yang sangat sering muncul, sehingga kata-kata yang sebenarnya kuat menandakan spam bisa kehilangan bobotnya.

### Kesimpulan

Pada dataset `spam.csv` dengan pembagian data 80:20 dan `random_state=50`, **CountVectorizer dengan stop_words bahasa Inggris lebih baik daripada TF-IDF** untuk klasifikasi spam menggunakan Multinomial Naive Bayes. CountVectorizer unggul pada akurasi, recall spam, dan F1-score, sehingga lebih tepat digunakan jika tujuan utamanya adalah meminimalkan spam yang lolos. TF-IDF hanya unggul pada precision spam, tetapi dengan konsekuensi banyak spam yang tidak terdeteksi.